# Model Playground

Interactive exploration of trained IMPALA agents on Memory Maze.

**What you'll do**:
1. List available checkpoints
2. Load a trained model
3. Run evaluation episodes with inline video
4. Plot learning curves from training logs
5. Compare multiple runs
6. Inspect model internals (LSTM hidden states)

**Prerequisite**: A trained checkpoint (from training or CLI)

In [ ]:
%matplotlib inline

In [ ]:
import sys, os, glob, time

import pathlib
PROJECT_ROOT = str(pathlib.Path.cwd().parent) if pathlib.Path('../train_impala.py').exists() else str(pathlib.Path.cwd())
sys.path.insert(0, PROJECT_ROOT)
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'notebooks'))
if sys.platform == "linux":
    os.environ.setdefault("MUJOCO_GL", "egl")
    os.environ.setdefault("PYOPENGL_PLATFORM", "egl")
else:
    os.environ.setdefault("MUJOCO_GL", "glfw")

import gym
import numpy as np
import torch
import matplotlib.pyplot as plt
from IPython.display import Video, display

from utils import (load_training_logs, logs_to_arrays, plot_learning_curve,
                    smooth, record_episode, save_video, display_video_in_notebook)

LOGDIR = os.path.expanduser('~/logs/torchbeast')
print(f'Log directory: {LOGDIR}')


def make_env(env_id, **kwargs):
    """Create env with EGL fallback — if MuJoCo fails due to EGL conflict, use Genesis."""
    try:
        return gym.make(env_id, **kwargs)
    except Exception as e:
        err_str = str(e)
        if "EGL" in err_str or "GLFWError" in err_str or "gladLoadGL" in err_str:
            genesis_id = env_id.replace("-v0", "-Genesis-v0")
            print(f"MuJoCo rendering unavailable, using Genesis: {genesis_id}")
            return gym.make(genesis_id, use_batch_renderer=False, **kwargs)
        raise

## 1. List Available Checkpoints

In [ ]:
checkpoints = sorted(glob.glob(os.path.join(LOGDIR, '*/model.tar')))

if not checkpoints:
    print(f'No checkpoints found in {LOGDIR}/*/')
    print('Run training first.')
else:
    print(f'Found {len(checkpoints)} checkpoint(s):\n')
    for cp in checkpoints:
        xpid = os.path.basename(os.path.dirname(cp))
        size_mb = os.path.getsize(cp) / 1e6
        ck = torch.load(cp, map_location='cpu', weights_only=False)
        steps = ck.get('step', '?')
        if isinstance(steps, (int, float)):
            steps = f'{int(steps):,}'
        print(f'  {xpid:40s}  IMPALA  {steps:>12s} steps  {size_mb:.1f} MB')

## 2. Load Checkpoint

Set `XPID` below to match the experiment you want to explore.

In [ ]:
# ── Set this to your experiment ID ──
XPID = 'notebook_impala_9x9'  # change to match your training run
# ────────────────────────────────────

checkpoint_path = os.path.join(LOGDIR, XPID, 'model.tar')
assert os.path.exists(checkpoint_path), f'Checkpoint not found: {checkpoint_path}'

checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)

print(f'Loaded: {checkpoint_path}')
print(f'Checkpoint keys: {list(checkpoint.keys())}')
print(f'Training steps: {checkpoint.get("step", "?"):,}')

In [ ]:
from train_impala import MemoryMazeNet as Net

model = Net(observation_shape=(3, 64, 64), num_actions=6)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f'IMPALA MemoryMazeNet loaded')

total_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {total_params:,}')

## 3. Run Evaluation Episodes

In [ ]:
ENV_ID = 'memory_maze:MemoryMaze-9x9-v0'
NUM_EVAL_EPISODES = 5

env = make_env(ENV_ID, disable_env_checker=True)
returns = []
all_actions = []

for ep in range(NUM_EVAL_EPISODES):
    obs = env.reset()
    episode_return = 0.0
    episode_actions = []
    done = False

    core_state = model.initial_state(batch_size=1)
    last_action = torch.zeros(1, dtype=torch.long)
    reward_t = torch.zeros(1)

    while not done:
        obs_t = torch.from_numpy(obs).unsqueeze(0).unsqueeze(0)
        inputs = {
            'frame': obs_t,
            'last_action': last_action.view(1, 1),
            'reward': reward_t.view(1, 1),
        }
        with torch.no_grad():
            outputs, core_state = model(inputs, core_state)
        action_idx = outputs['action'].item()
        episode_actions.append(action_idx)

        obs, reward, done, info = env.step(action_idx)
        episode_return += reward
        last_action = torch.tensor([action_idx])
        reward_t = torch.tensor([reward])

    returns.append(episode_return)
    all_actions.extend(episode_actions)
    print(f'  Episode {ep+1}: return={episode_return:.1f}, steps={len(episode_actions)}')

print(f'\nMean return: {np.mean(returns):.1f} +/- {np.std(returns):.1f}')

## 4. Record Evaluation Video

In [ ]:
def make_impala_policy(model):
    core_state = model.initial_state(batch_size=1)
    last_action = torch.zeros(1, dtype=torch.long)

    def policy(obs, reward=0.0):
        nonlocal core_state, last_action
        reward_t = torch.tensor([reward])
        obs_t = torch.from_numpy(obs).unsqueeze(0).unsqueeze(0)
        inputs = {'frame': obs_t, 'last_action': last_action.view(1, 1), 'reward': reward_t.view(1, 1)}
        with torch.no_grad():
            outputs, core_state_new = model(inputs, core_state)
        core_state = core_state_new
        action = outputs['action'].item()
        last_action = torch.tensor([action])
        return action
    return policy


policy_fn = make_impala_policy(model)

env_rec = make_env(ENV_ID, disable_env_checker=True)
frames, total_ret, n_steps = record_episode(env_rec, policy_fn=policy_fn, max_steps=1000)
save_video(frames, 'eval_episode.mp4', fps=10)
env_rec.close()

print(f'Recorded {len(frames)} frames, return={total_ret:.1f}')
display_video_in_notebook('eval_episode.mp4')

## 5. Learning Curves from Training Logs

In [ ]:
try:
    rows = load_training_logs(XPID, logdir=LOGDIR)
    step_key = 'step'
    steps, rets = logs_to_arrays(rows, step_key=step_key, value_key='mean_episode_return')

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Learning curve
    plot_learning_curve(steps, rets, label=XPID, smooth_window=20, ax=axes[0],
                        title=f'Learning Curve — IMPALA')

    # Loss
    _, total_loss = logs_to_arrays(rows, value_key='total_loss')
    if len(total_loss) > 0:
        axes[1].plot(steps[:len(total_loss)], smooth(total_loss, 20), linewidth=1.5)
        axes[1].set_title('Total Loss')
        axes[1].set_xlabel('Steps')
        axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
except FileNotFoundError:
    print(f'No logs.csv found for {XPID}')

## 6. Compare Multiple Runs

Overlay learning curves from different experiments.

In [ ]:
# List all experiments with logs
XPIDS_TO_COMPARE = []  # add your xpids here, e.g. ['impala_mujoco', 'impala_genesis']

# Auto-detect if empty
if not XPIDS_TO_COMPARE:
    for d in sorted(os.listdir(LOGDIR)):
        if os.path.exists(os.path.join(LOGDIR, d, 'logs.csv')):
            XPIDS_TO_COMPARE.append(d)

if len(XPIDS_TO_COMPARE) < 2:
    print(f'Need at least 2 experiments to compare. Found: {XPIDS_TO_COMPARE}')
else:
    fig, ax = plt.subplots(figsize=(12, 5))
    colors = plt.cm.tab10.colors

    for i, xpid in enumerate(XPIDS_TO_COMPARE):
        try:
            rows = load_training_logs(xpid, logdir=LOGDIR)
            steps, rets = logs_to_arrays(rows)
            plot_learning_curve(steps, rets, label=xpid, color=colors[i % len(colors)],
                                smooth_window=20, ax=ax)
        except Exception as e:
            print(f'  {xpid}: {e}')

    ax.set_title('Run Comparison')
    ax.legend(fontsize=10)
    plt.tight_layout()
    plt.show()

## 7. Action Distribution

Histogram of actions taken during evaluation episodes.

In [ ]:
ACTION_NAMES = ['Noop', 'Forward', 'Left', 'Right', 'Fwd+Left', 'Fwd+Right']

fig, ax = plt.subplots(figsize=(8, 4))
counts = np.bincount(all_actions, minlength=6)
ax.bar(range(6), counts / counts.sum() * 100, color='tab:blue', edgecolor='white')
ax.set_xticks(range(6))
ax.set_xticklabels(ACTION_NAMES, rotation=30)
ax.set_ylabel('Frequency (%)')
ax.set_title(f'Action Distribution (IMPALA, {len(all_actions)} actions)')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 8. Inspect Model Internals

LSTM hidden state norms over an episode — tracks how the recurrent memory evolves.

In [ ]:
env_inspect = make_env(ENV_ID, disable_env_checker=True)
obs = env_inspect.reset()
done = False
hidden_norms = []

core_state = model.initial_state(batch_size=1)
last_action = torch.zeros(1, dtype=torch.long)
reward_t = torch.zeros(1)

while not done:
    obs_t = torch.from_numpy(obs).unsqueeze(0).unsqueeze(0)
    inputs = {'frame': obs_t, 'last_action': last_action.view(1, 1), 'reward': reward_t.view(1, 1)}
    with torch.no_grad():
        outputs, core_state = model(inputs, core_state)
    hidden_norms.append(core_state[0].norm().item())
    action_idx = outputs['action'].item()

    obs, reward, done, info = env_inspect.step(action_idx)
    last_action = torch.tensor([action_idx])
    reward_t = torch.tensor([reward])

env_inspect.close()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(hidden_norms, linewidth=1.5)
ax.set_xlabel('Step')
ax.set_ylabel('Norm')
ax.set_title('LSTM Hidden State Norm')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Cross-Backend Evaluation

Load a checkpoint trained on one backend and evaluate on another.
This tests whether learned policies transfer between MuJoCo and Genesis rendering.

In [ ]:
CROSS_ENV_IDS = [
    ('MuJoCo', 'memory_maze:MemoryMaze-9x9-v0'),
    # Uncomment if Genesis is available:
    # ('Genesis', 'memory_maze:MemoryMaze-9x9-Genesis-v0'),
]

for name, env_id in CROSS_ENV_IDS:
    env_cross = make_env(env_id, disable_env_checker=True)
    cross_returns = []

    for ep in range(3):
        obs = env_cross.reset()
        ep_return = 0.0
        done = False

        core_state = model.initial_state(1)
        last_a = torch.zeros(1, dtype=torch.long)
        rew = torch.zeros(1)
        while not done:
            obs_t = torch.from_numpy(obs).unsqueeze(0).unsqueeze(0)
            inp = {'frame': obs_t, 'last_action': last_a.view(1,1), 'reward': rew.view(1,1)}
            with torch.no_grad():
                out, core_state = model(inp, core_state)
            a = out['action'].item()
            obs, r, done, _ = env_cross.step(a)
            ep_return += r
            last_a = torch.tensor([a])
            rew = torch.tensor([r])

        cross_returns.append(ep_return)

    env_cross.close()
    print(f'{name:10s}  mean={np.mean(cross_returns):.1f} +/- {np.std(cross_returns):.1f}  '
          f'({cross_returns})')

## Next Steps

- **Environment exploration**: See `01_environment_tour.ipynb`
- **Paper comparison**: See Table 3 in the Memory Maze paper for reference scores